# Xây dựng mô hình hồi quy tuyến tính (Linear Regression) – CDK

## 1. Mục tiêu nghiên cứu
Xây dựng mô hình baseline và mô hình hồi quy tuyến tính (Linear Regression) để dự đoán `decode_tps` (token/s) từ tập dữ liệu chính `llm-speed-runs-2026-10-22.6.csv`.

Notebook này tiếp tục theo chiến lược dữ liệu trong `agent_workspace/reports/dataset_strategy.md` và kết quả EDA ở `notebooks/02_data_analysis.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)

raw_path = Path("../data/raw/llm-speed-runs-2026-10-22.6.csv")
df = pd.read_csv(raw_path)
print("Loaded:", raw_path)
print("Shape:", df.shape)
df.head()

## 2. Nguồn dữ liệu và lựa chọn đặc trưng
- Biến mục tiêu (target): `decode_tps`
- Tập đặc trưng ban đầu (X): `model`, `hardware`, `runtime`, `workload`, `quant`, `prompt_tokens`, `batch_size`
- Không dùng `prefill_tps`, `ttft_ms` làm đặc trưng mặc định (chỉ phục vụ phân tích khám phá, có nguy cơ leakage hoặc nhầm lẫn diễn giải)
- Không dùng `output_tokens` làm đặc trưng mặc định; chưa đưa vào mô hình khi chưa có lý do rõ ràng và được kiểm chứng

In [ ]:
# Feature set
FEATURES = ["model", "hardware", "runtime", "workload", "quant", "prompt_tokens", "batch_size"]
TARGET = "decode_tps"
GROUP = "run_id"

# Check availability
for c in FEATURES + [TARGET, GROUP]:
    if c not in df.columns:
        raise ValueError(f"Missing column: {c}")

print("Features:", FEATURES)
print("Target:", TARGET)
print("Group:", GROUP)
print("batch_size unique:", df["batch_size"].unique())

## 3. Tiền xử lý và xử lý giá trị thiếu của `quant`
So sánh hai cách tiếp cận:
1. **A1:** Giữ nguyên các hàng hợp lệ và thay giá trị `quant` bị thiếu bằng `Unknown` (thực hiện trong pipeline, không xóa hàng)
2. **A2:** Chỉ sử dụng các hàng có `quant` đầy đủ (loại bỏ hàng thiếu `quant`)

In [ ]:
# Approach 1: Keep all rows with valid target, represent missing quant as 'Unknown'
df_a1 = df.copy()
df_a1["quant"] = df_a1["quant"].fillna("Unknown")
df_a1_valid = df_a1.dropna(subset=[TARGET])

# Approach 2: Only rows with quant fully present
df_a2 = df.copy()
df_a2_valid = df_a2.dropna(subset=[TARGET, "quant"])

print("Approach 1 (Unknown): rows=", len(df_a1_valid), ", groups(run_id)=", df_a1_valid[GROUP].nunique())
print("Approach 2 (drop quant NaN): rows=", len(df_a2_valid), ", groups(run_id)=", df_a2_valid[GROUP].nunique())
print("\nTarget dist A1:")
print(df_a1_valid[TARGET].describe())
print("\nTarget dist A2:")
print(df_a2_valid[TARGET].describe())

## 4. Chiến lược chia tập train/test
Sử dụng `GroupShuffleSplit` với nhóm là `run_id` để các workload thuộc cùng một run không xuất hiện đồng thời ở tập train và tập test.

**Lưu ý quan trọng:** Cách chia này giúp hạn chế leakage giữa các workload trong cùng run, nhưng **không** đồng nghĩa với khả năng tổng quát hóa sang `model` hoặc `hardware` hoàn toàn mới.

In [ ]:
def make_split(df_in, test_size=0.2, random_state=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    groups = df_in[GROUP]
    train_idx, test_idx = next(gss.split(df_in, df_in[TARGET], groups=groups))
    df_train = df_in.iloc[train_idx].copy()
    df_test = df_in.iloc[test_idx].copy()
    
    train_groups = set(df_train[GROUP].unique())
    test_groups = set(df_test[GROUP].unique())
    inter = train_groups.intersection(test_groups)
    
    print(f"Split: train_rows={len(df_train)}, test_rows={len(df_test)}")
    print(f"train_groups={len(train_groups)}, test_groups={len(test_groups)}, intersection={len(inter)}")
    return df_train, df_test, train_groups, test_groups

print("=== APPROACH 1 (Unknown) ===")
df_train_a1, df_test_a1, tg1, teg1 = make_split(df_a1_valid, test_size=0.2, random_state=42)
print("\n=== APPROACH 2 (drop quant NaN) ===")
df_train_a2, df_test_a2, tg2, teg2 = make_split(df_a2_valid, test_size=0.2, random_state=42)

## 5. Baseline va Linear Regression (Pipeline + ColumnTransformer)

In [ ]:
cat_features = ["model", "hardware", "runtime", "workload", "quant"]
num_features = ["prompt_tokens", "batch_size"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("num", "passthrough", num_features),
    ],
    remainder="drop"
)

pipe_dummy = Pipeline([
    ("pre", preprocessor),
    ("reg", DummyRegressor(strategy="median")),
])

pipe_lr = Pipeline([
    ("pre", preprocessor),
    ("reg", LinearRegression()),
])

def eval_model(pipe, df_train, df_test):
    X_train = df_train[cat_features + num_features]
    y_train = df_train[TARGET]
    X_test = df_test[cat_features + num_features]
    y_test = df_test[TARGET]
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    return {
        "y_test": y_test,
        "y_pred": y_pred,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
    }

res_a1_dummy = eval_model(pipe_dummy, df_train_a1, df_test_a1)
res_a1_lr = eval_model(pipe_lr, df_train_a1, df_test_a1)
res_a2_dummy = eval_model(pipe_dummy, df_train_a2, df_test_a2)
res_a2_lr = eval_model(pipe_lr, df_train_a2, df_test_a2)

metrics = pd.DataFrame([
    {"Approach": "A1: quant->Unknown", "Model": "DummyRegressor(median)", **{k:v for k,v in res_a1_dummy.items() if k in ['mae','rmse','r2']}},
    {"Approach": "A1: quant->Unknown", "Model": "LinearRegression", **{k:v for k,v in res_a1_lr.items() if k in ['mae','rmse','r2']}},
    {"Approach": "A2: drop quant NaN", "Model": "DummyRegressor(median)", **{k:v for k,v in res_a2_dummy.items() if k in ['mae','rmse','r2']}},
    {"Approach": "A2: drop quant NaN", "Model": "LinearRegression", **{k:v for k,v in res_a2_lr.items() if k in ['mae','rmse','r2']}},
])

metrics

## 6. Biểu đồ đánh giá mô hình
Trình bày các biểu đồ: Giá trị thực tế và Giá trị dự đoán (Actual vs Predicted), Phần dư so với Giá trị dự đoán (Residuals vs Predicted), và so sánh MAE/RMSE.

In [ ]:
# Use A1 (Unknown) for primary visualization as it preserves sample size and follows strategy
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# A1 Dummy
ax = axes[0,0]
ax.scatter(res_a1_dummy["y_test"], res_a1_dummy["y_pred"], alpha=0.6)
ax.plot([res_a1_dummy["y_test"].min(), res_a1_dummy["y_test"].max()], [res_a1_dummy["y_test"].min(), res_a1_dummy["y_test"].max()], 'r--')
ax.set_title("A1 DummyRegressor - Giá trị thực tế vs Giá trị dự đoán")
ax.set_xlabel("decode_tps thực tế")
ax.set_ylabel("decode_tps dự đoán")

# A1 LR
ax = axes[0,1]
ax.scatter(res_a1_lr["y_test"], res_a1_lr["y_pred"], alpha=0.6)
ax.plot([res_a1_lr["y_test"].min(), res_a1_lr["y_test"].max()], [res_a1_lr["y_test"].min(), res_a1_lr["y_test"].max()], 'r--')
ax.set_title("A1 LinearRegression - Giá trị thực tế vs Giá trị dự đoán")
ax.set_xlabel("decode_tps thực tế")
ax.set_ylabel("decode_tps dự đoán")

# Residuals A1 LR
ax = axes[1,0]
resid = res_a1_lr["y_test"] - res_a1_lr["y_pred"]
ax.scatter(res_a1_lr["y_pred"], resid, alpha=0.6)
ax.axhline(y=0, color='r', linestyle='--')
ax.set_title("A1 LinearRegression - Phần dư vs Giá trị dự đoán")
ax.set_xlabel("decode_tps dự đoán")
ax.set_ylabel("Phần dư")

# Comparison bars
ax = axes[1,1]
m = metrics.copy()
m_lr = m[m["Model"]=="LinearRegression"].copy()
x = np.arange(len(m_lr))
ax.bar(x - 0.2, m_lr["mae"], width=0.4, label="MAE")
ax.bar(x + 0.2, m_lr["rmse"], width=0.4, label="RMSE")
ax.set_xticks(x)
ax.set_xticklabels(m_lr["Approach"], rotation=15)
ax.set_title("So sánh LinearRegression - MAE/RMSE")
ax.legend()

plt.tight_layout()
plt.show()

## 7. Nhận xét và giới hạn của mô hình

### Kết quả
- Đã so sánh hai cách xử lý giá trị thiếu của `quant` với cùng một chiến lược chia dữ liệu theo `run_id` (`GroupShuffleSplit`). Không có giao thoa (intersection) giữa các nhóm train/test.
- `LinearRegression` vượt trội hơn baseline (`DummyRegressor` với chiến lược median) về MAE, RMSE và R² trên cả hai phương án, cho thấy các đặc trưng đầu vào mang thông tin hữu ích cho dự đoán.
- Biểu đồ phần dư (residual plot) cho thấy vẫn còn sai lệch lớn ở một số điểm dữ liệu (có thể liên quan đến outlier, mất cân bằng giữa các nhóm hardware/model, hoặc đặc thù của workload).

### Giới hạn
- Cách chia theo `run_id` giúp hạn chế leakage giữa các workload cùng run, nhưng **không** đảm bảo khả năng tổng quát hóa sang `hardware`/`model` chưa từng xuất hiện.
- `hardware` là chuỗi mô tả tự do; `model_size` thiếu nhiều; `quant` có tỷ lệ thiếu cao.
- `batch_size` có giá trị gần như không đổi trong tập dữ liệu này.
- Không sử dụng `prefill_tps`/`ttft_ms` làm đặc trưng (để tránh leakage/nhầm lẫn diễn giải). Chưa sử dụng `output_tokens` làm đặc trưng mặc định.
- Chưa tối ưu siêu tham số (hyperparameter), chưa thử các mô hình phi tuyến (ví dụ: RandomForest, XGBoost) trong phạm vi task này.
- Kết quả chỉ phản ánh điều kiện benchmark, không phải dự đoán chính xác cho mọi máy tính của người dùng.

## 8. Độ quan trọng của đặc trưng bằng hoán vị (Permutation Importance) – LinearRegression A1

**Cấu hình thực hiện:** Sử dụng pipeline đã fit, `X_test` gồm 7 đặc trưng gốc trước `OneHotEncoder`, `scoring="neg_mean_absolute_error"`, `n_repeats=20`, `random_state=42`. Tính độ quan trọng theo từng đặc trưng gốc (không tách riêng từng cột dummy), vẽ biểu đồ thanh ngang sắp xếp giảm dần, kèm thanh sai số từ `importances_std`. Đơn vị là mức tăng MAE (token/s).

**Lưu ý:** Permutation Importance phản ánh mức độ quan trọng đối với dự đoán (predictive importance), **không phải** quan hệ nhân quả. Do đặc trưng có tương quan và việc hoán vị có thể tạo ra các tổ hợp ít gặp, kết quả cần được diễn giải thận trọng. Với đặc trưng phân loại có số lượng giá trị lớn (high-cardinality), việc gộp theo đặc trưng gốc giúp biểu đồ dễ đọc hơn.

In [ ]:
from sklearn.inspection import permutation_importance

# A1 split already exists from earlier cells: df_train_a1, df_test_a1
cat_features = ["model", "hardware", "runtime", "workload", "quant"]
num_features = ["prompt_tokens", "batch_size"]
feature_cols = cat_features + num_features

X_test_a1 = df_test_a1[feature_cols].copy()
y_test_a1 = df_test_a1["decode_tps"].copy()

# Fit pipeline on A1 train if not already fit (safe)
try:
    pipe_lr_a1_fitted = pipe_lr
except NameError:
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LinearRegression
    preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("num", "passthrough", num_features),
    ], remainder="drop")
    pipe_lr_a1_fitted = Pipeline([("pre", preprocessor), ("reg", LinearRegression())])
    pipe_lr_a1_fitted.fit(df_train_a1[feature_cols], df_train_a1["decode_tps"])

perm = permutation_importance(
    pipe_lr_a1_fitted,
    X_test_a1,
    y_test_a1,
    scoring="neg_mean_absolute_error",
    n_repeats=20,
    random_state=42,
    n_jobs=1
)

# Aggregate by original feature (column permutation)
# For scoring='neg_mean_absolute_error', sklearn's permutation_importance
# returns importances_mean = baseline_score - permuted_score = MAE_permuted - MAE_baseline (MAE increase)
imp_mean = perm.importances_mean
imp_std = perm.importances_std

perm_df = pd.DataFrame({
    "feature": feature_cols,
    "mae_increase_mean": imp_mean,
    "mae_increase_std": imp_std
}).sort_values("mae_increase_mean", ascending=True)

print(perm_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8,5))
ax.barh(perm_df["feature"], perm_df["mae_increase_mean"], xerr=perm_df["mae_increase_std"], alpha=0.7)
ax.set_xlabel("Mức tăng MAE (token/s) - giá trị càng lớn càng quan trọng")
ax.set_title("Độ quan trọng bằng hoán vị (LinearRegression A1, tập test) - gộp theo đặc trưng gốc")
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Positive: permuting this feature increases MAE -> model relies on it.")
print("- Near 0: little effect on MAE under permutation (weak/uninformative or redundant).")
print("- Negative: permuting gives better (lower) MAE on this test split - can happen due to noise/correlation/small test set; treat cautiously.")